# Scenario 02 — Task Cancellation

A client publishes a task, then cancels it before any provider attempts. After cancellation
the task status flips to `CANCELED` and attempts to cancel again revert.

**Role:** Client only.

**Prerequisites:**
- `CLIENT_PRIVATE_KEY`
- An existing source address to publish against (scenario 01 produced one, or use any active source)

**Flow:**

```
Client
  │
  ├─ 1. publish_task       → NEW
  ├─ 2. task.cancel()      → Receipt
  └─ 3. get_status()        → CANCELED
```


## 0. Setup


In [ ]:
import os
import time

from web3 import Web3

from ogpu.client import (
    ChainConfig,
    ChainId,
    DeliveryMethod,
    ImageEnvironments,
    SourceInfo,
    TaskInfo,
    TaskInput,
    publish_source,
    publish_task,
)
from ogpu.protocol import Master, Provider, Response, Source, Task, vault

ChainConfig.set_chain(ChainId.OGPU_TESTNET)

CLIENT_KEY = os.environ["CLIENT_PRIVATE_KEY"]


In [ ]:
SOURCE_ADDR = "0xYOUR_ACTIVE_SOURCE_ADDRESS"  # replace with real source


## 1. Publish a task


In [ ]:
task_info = TaskInfo(
    source=SOURCE_ADDR,
    config=TaskInput(function_name="some_function", data={"prompt": "cancel me"}),
    expiryTime=int(time.time()) + 3600,
    payment=Web3.to_wei(0.01, "ether"),
)

task = publish_task(task_info, private_key=CLIENT_KEY)
print(f"Task: {task.address}")
print(f"Status: {task.get_status()}")


## 2. Cancel

`Task.cancel()` is a thin wrapper over `Controller.cancelTask(task)`. Returns a `Receipt`.
Only works while the task is still in `NEW` state — once a provider has attempted, cancel reverts.


In [ ]:
receipt = task.cancel(signer=CLIENT_KEY)
print(f"Cancel tx: {receipt.tx_hash}")


## 3. Verify


In [ ]:
from ogpu.types import TaskStatus

status = task.get_status()
print(f"Task status: {status}")
assert status == TaskStatus.CANCELED


## 4. (Bonus) Try cancelling twice

The second cancel should revert with a typed exception — the SDK decodes the revert
reason into an `OGPUError` subclass rather than a raw `ContractLogicError`.


In [ ]:
from ogpu.types import OGPUError

try:
    task.cancel(signer=CLIENT_KEY)
except OGPUError as e:
    print(f"Second cancel reverted as expected: {type(e).__name__}: {e}")
